# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{getattr(metadata, 'name', '')}: {getattr(metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs.

We will enumerate all record sets (`@type='RecordSet'`), their fields, and the corresponding column `@id`s.

In [ ]:
from typing import Any

def get_schema_entities(metadata: Any):
    """
    Traverse the metadata JSON-LD dict to extract record sets, fields, and columns with their @ids.
    """
    import requests
    import json
    # Download the Croissant JSON
    response = requests.get(url)
    schema_dict = response.json()
    
    # Helper functions
    def find_by_type(obj, target_type):
        results = []
        if isinstance(obj, dict):
            if obj.get('@type') == target_type:
                results.append(obj)
            for v in obj.values():
                results.extend(find_by_type(v, target_type))
        elif isinstance(obj, list):
            for item in obj:
                results.extend(find_by_type(item, target_type))
        return results

    record_sets = find_by_type(schema_dict, 'RecordSet')
    record_sets_info = []
    for rec in record_sets:
        rec_id = rec.get('@id')
        rec_name = rec.get('name', rec_id)
        # Each record set may have 'field' as list of field/@id
        # Let's resolve references if possible
        fields = rec.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        field_objs = []
        for field in fields:
            if isinstance(field, str):
                # Try to find the field object
                field_obj = next((f for f in find_by_type(schema_dict, 'Field') if f.get('@id')==field), None)
                if field_obj:
                    field_objs.append(field_obj)
                else:
                    field_objs.append({'@id':field})
            elif isinstance(field, dict):
                if 'cr:Field' in field.get('@type', ''):
                    field_objs.append(field)
                elif '@id' in field:
                    field_obj = next((f for f in find_by_type(schema_dict, 'Field') if f.get('@id')==field['@id']), None)
                    if field_obj:
                        field_objs.append(field_obj)
                    else:
                        field_objs.append(field)
        # For each field, try to show its columns
        fields_info = []
        for fld in field_objs:
            f_id = fld.get('@id')
            f_name = fld.get('name', f_id)
            columns = fld.get('column', [])
            if isinstance(columns, dict):
                columns = [columns]
            col_ids = []
            for col in columns:
                if isinstance(col, str):
                    col_ids.append(col)
                elif isinstance(col, dict):
                    col_ids.append(col.get('@id', str(col)))
            fields_info.append({'@id': f_id, 'name': f_name, 'columns': col_ids})
        record_sets_info.append({'@id': rec_id, 'name': rec_name, 'fields': fields_info})
    return record_sets_info

record_sets_info = get_schema_entities(metadata)
if not record_sets_info:
    print("No RecordSet entities found in the Croissant metadata.")
else:
    for rec in record_sets_info:
        print(f"Record Set: {rec['name']} (@id={rec['@id']})")
        for field in rec['fields']:
            print(f"  Field: {field['name']} (@id={field['@id']})")
            for col in field['columns']:
                print(f"    Column: {col}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# For demonstration, select all RecordSets found
record_sets = [rec['@id'] for rec in record_sets_info]

if not record_sets:
    print("No record sets found to extract.")
else:
    dataframes = {}
    for rec_id in record_sets:
        try:
            records = list(dataset.records(record_set=rec_id))
            if len(records) == 0:
                print(f"No records found for RecordSet @id={rec_id}")
                continue
            dataframes[rec_id] = pd.DataFrame(records)
            print(f"Loaded DataFrame for RecordSet @id={rec_id} with columns: {dataframes[rec_id].columns.tolist()}")
        except Exception as e:
            print(f"Failed to load records for RecordSet @id={rec_id}: {e}")

# Pick one RecordSet ID for further demonstration if available
if dataframes:
    demo_record_set_id = next(iter(dataframes.keys()))
    print(f"Using RecordSet @id={demo_record_set_id} for demonstration")
    df = dataframes[demo_record_set_id]
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This may include operations like removing outliers, transforming distributions, or grouping by key fields.

In [ ]:
import numpy as np

# Example: choose a numeric field from the data for EDA
numeric_field = None
group_field = None
if dataframes:
    df = dataframes[demo_record_set_id]
    # Try to automatically detect a numeric field (float/int)
    num_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if num_candidates:
        numeric_field = num_candidates[0]
    # Try to select a group field (categorical-like)
    cat_candidates = df.select_dtypes(include=['object', 'category']).columns.tolist()
    if cat_candidates:
        group_field = cat_candidates[0]

    if numeric_field is None:
        print("No numeric field found for EDA.")
    else:
        # Filtering
        threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

        # Grouping
        if group_field and group_field in df.columns:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field} (showing average of numeric columns):")
            display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field:
    # Histogram of the numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.show()

    # If group_field is available, boxplot
    if group_field and group_field in df.columns:
        plt.figure(figsize=(12,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated loading a Croissant-structured dataset using the `mlcroissant` library. We explored its record sets, extracted tabular data, performed basic filtering and normalization, and visualized the results. This approach allows for reproducible, schema-driven FAIR data science.

**Next Steps:**
- Explore additional record sets or fields by their `@id`
- Use domain-specific EDA methods for proteomics/mass spectrometry data
- Apply ML and statistical models using the clean DataFrames